# PyWake P2X Example

This notebook reproduces the example shown in `hydesign/use_cases/hpp_pywake_p2x.py`.
A hybrid power plant with wind, solar, storage and power-to-gas is evaluated 
using a PyWake based wind farm model, and selected outputs are plotted.

In [ ]:
import numpy as np
import pandas as pd
from hydesign.use_cases.hpp_pywake_p2x import hpp_model
from hydesign.examples import examples_filepath
from py_wake.examples.data.hornsrev1 import Hornsrev1Site
from py_wake.turbulence_models.stf import STF2017TurbulenceModel
from py_wake import NOJ
from py_wake.deflection_models import JimenezWakeDeflection
from py_wake.examples.data.dtu10mw_surrogate import DTU10MW_1WT_Surrogate
from py_wake.superposition_models import LinearSum
from topfarm.utils import regular_generic_layout
from scipy.spatial import ConvexHull
import matplotlib.pyplot as plt

## Setting up the model

In [ ]:
name = 'Denmark_good_wind'
examples_sites = pd.read_csv(f'{examples_filepath}examples_sites.csv', index_col=0, sep=';')
ex_site = examples_sites.loc[examples_sites.name == name]

longitude = ex_site['longitude'].values[0]
latitude = ex_site['latitude'].values[0]
altitude = ex_site['altitude'].values[0]
sim_pars_fn = examples_filepath + ex_site['sim_pars_fn'].values[0]
input_ts_fn = examples_filepath + ex_site['input_ts_fn'].values[0]

intervals_per_hour = 1
n_wt = 40
n_loads = 4
wt = DTU10MW_1WT_Surrogate()
d = wt.diameter()
site = Hornsrev1Site()
sx = 4 * d
sy = 5 * d
x, y = regular_generic_layout(n_wt, sx, sy, stagger=0, rotation=0)
N_ws = 365 * 24 * intervals_per_hour
time_stamp = np.arange(N_ws) / 6 / 24
farm = NOJ(site, wt, turbulenceModel=STF2017TurbulenceModel(),
    deflectionModel=JimenezWakeDeflection(), superpositionModel=LinearSum())
yaw = 30 * np.sin(np.arange(N_ws) / 100)
tilt = np.zeros(N_ws)

hpp = hpp_model(latitude=latitude,
                longitude=longitude,
                altitude=altitude,
                sim_pars_fn=sim_pars_fn,
                input_ts_fn=input_ts_fn,
                intervals_per_hour=intervals_per_hour,
                farm=farm,
                x=x,
                y=y,
                tilt=tilt,
                time_stamp=time_stamp,
                n_loads=n_loads,
                H2_demand_fn=6000)

## Evaluate the design

In [ ]:
p_rated = 10.0
points = np.asarray([x, y]).T
hull = ConvexHull(points)
area = hull.volume * 1e-6
wind_MW_per_km2 = 3
clearance = wt.hub_height() - d / 2
sp = 360

inputs = dict(clearance=clearance, sp=sp, p_rated=p_rated, Nwt=n_wt, wind_MW_per_km2=wind_MW_per_km2,
              solar_MW=200, surface_tilt=45, surface_azimuth=180, DC_AC_ratio=1.5,
              b_P=40, b_E_h=4, cost_of_battery_P_fluct_in_peak_price_ratio=5,
              ptg_MW=800, HSS_kg=5000,
              yaw=yaw)

outs = hpp.evaluate(**inputs)
hpp.print_design(list(inputs.values()), outs)

## Plotting example output

In [ ]:
sensors = wt.loadFunction.output_keys
sensor_no = 0
turb = 0
plt.figure()
plt.plot(hpp.prob['loads_rel_ext'][sensor_no, turb, :])
plt.xlabel('Time [h]')
plt.ylabel('Load ratio [-]')
plt.title(f'Extreme loads for {sensors[sensor_no]}')
plt.show()

The above figure displays the relative extreme load time series for one wind turbine sensor.